# Zero-shot — **LaGoVAD released checkpoint** on DoTA-CAP-dev, TAD, DADA-2000 (beside W0 / R4)

Rule (written before any read): `core/docs/final/ZS_TAD_DADA.md` **§7**. Prerequisites on Drive: W0 / R4 reads of
DoTA-CAP-dev (`wu_arms.ipynb`, `stp_nexar_dotacap.ipynb`) and of TAD / DADA (`zs_tad_dada_r4.ipynb`).

| row | what | input |
|---|---|---|
| **LaGoVAD** | released `best.ckpt`, KIP off, released prompts | plain CLIP `_ncc` rows (= the arms' own CLIP columns, checked) |
| W0 | Nexar host, no SA², s2024 (already read) | A2 bake |
| R4 | W0 + SA², s2024 (already read) | A2 bake + patch cache |

| step | what | GPU? |
|---|---|---|
| 1–3 | setup, deps, helpers | no |
| 4 | ids + preflight (checkpoint, caches, the arms' reads) | no |
| 5 | **integrity**: plain CLIP rows == the A2 bakes' first 512 columns on every item (§7.3-1) | no |
| 6 | DoTA-CAP-dev: `protocol_b_eval --baseline` at protocol **B** (s3) + **A** (s8, printed) | optional |
| 7 | TAD: `core.evaluate --baseline-ckpt` (= Gate T0, re-run here; §7.3-2) | optional |
| 8 | DADA-2000: `dada_whole_eval --baseline` (370 sources, s8) | optional |
| 9 | `zs_lagovad_readout` → **paste `REPORTS/zs_lagovad_readout.md` to Claude** | no |

CPU is enough (forward passes only; a GPU just makes it faster). Every step skips what is already on Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip, ckpts/best.ckpt
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: A2 bakes (only for the input check), Nexar_window defs
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code + the arms' reads + this notebook's outputs
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final-lagovad-zs there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'ZS_LAGOVAD_ARM'), 'this upload predates the LaGoVAD zero-shot build -- re-upload kat-vad/'
assert '## 7. LaGoVAD' in (REPO / 'core/docs/final/ZS_TAD_DADA.md').read_text(), 'no §7 (the rule) in this upload'

LAG, RUN_NAME = constants.ZS_LAGOVAD_ARM, constants.ZS_LAGOVAD_RUN     # 'LaGoVAD', 'released'
SEED     = constants.ZS_SEED                                           # 2024 (the arms)
CONTROL, SA2 = constants.ZS_CONTROL_ARM, constants.ZS_SA2_ARM          # W0, R4
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
T2NAME   = constants.DADA_ORIGIN_DATASET                               # DADA2000_orig
C        = constants.NEXAR_CONSTRUCTION_WINDOW

CKPT     = Path(DRIVE) / 'ckpts' / 'best.ckpt'                         # LaGoVAD released (Gate T0's file)
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
TAD      = constants.DATA_ROOT / constants.TAD_DATASET
TAD_NCC  = Path(DRIVE) / 'cache' / 'clip' / 'TAD_ncc'
T2       = constants.DATA_ROOT / T2NAME
T2_CLIP  = Path(DRIVE) / 'cache' / 'clip' / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
DEFS     = Path(DRIVE_V2) / 'data' / f'{constants.NEXAR_DATASET}_{C}'   # the arms' DADA definitions (defs.json)

V2_CACHE = Path(DRIVE_V2) / 'cache' / 'v2' / 'A2_S'                    # the arms' bakes: input check only
BAKE     = {'dota': V2_CACHE / f'{CAP_DEV}_s3_from_s1_nexar',
            'tad': V2_CACHE / f'{constants.TAD_DATASET}_s8_nexar',
            'dada': V2_CACHE / f'{T2NAME}_test_sources_nexar'}
PLAIN    = {'dota': DOTA_S1, 'tad': TAD_NCC, 'dada': T2_CLIP}
PLAIN_STRIDE = {'dota': constants.V2_E1_STRIDE_BC, 'tad': 1, 'dada': 1}   # DoTA s1[::3]; TAD / DADA caches are s8

OUT      = constants.OUTPUT_ROOT
ARM_DOTA = {CONTROL: OUT / constants.WU_OUTPUT_DIRNAME / 'eval' / 'dota' / CAP_DEV / CONTROL,
            SA2: OUT / constants.STP_OUTPUT_DIRNAME / 'eval' / 'dota' / CAP_DEV / SA2}
ZS_ROOT  = OUT / constants.ZS_OUTPUT_DIRNAME                            # eval/{tad,dada}/{W0,R4}
ROOT     = OUT / constants.ZS_LAGOVAD_OUTPUT_DIRNAME                    # this notebook
EVAL     = ROOT / 'eval'
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/zs_lagovad')
for p in (REPORTS, LOCAL):
    p.mkdir(parents=True, exist_ok=True)

for name, path in (('LaGoVAD best.ckpt', CKPT), ('DoTA s1', DOTA_S1), ('DoTA labels', DOTA / 'labels_s8'),
                   ('DoTA metadata', DOTA / 'metadata_val.json'), ('TAD labels', TAD / constants.FRAME_LABELS_TEST_FILENAME),
                   ('TAD_ncc', TAD_NCC), ('T2 meta', T2 / constants.META_FILENAME), ('T2 CLIP', T2_CLIP),
                   ('T2 census', T2_CENSUS), ('DADA annotation', ANNOT), ('Nexar defs', DEFS / 'defs.json'),
                   *[(f'{s} A2 bake', b) for s, b in BAKE.items()],
                   *[(f'{a} DoTA-CAP-dev read', d / 'clip_aucs.json') for a, d in ARM_DOTA.items()],
                   *[(f'{a} {s} read', ZS_ROOT / 'eval' / s / a) for s in ('tad', 'dada') for a in (CONTROL, SA2)]):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:22s} {path}')
print('outputs ->', ROOT)

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'no GPU -- fine, this notebook runs on CPU'

In [ ]:
import hashlib
import shutil
import subprocess
import time

import numpy as np

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    """Copy a Drive dir (only ``ids`` + manifests if given) to VM disk once (bit-identical; C2)."""
    dst.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10)."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for block in iter(lambda: fh.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

## 4. Ids + checkpoint fingerprint

In [ ]:
from core.data.v2_splits import load_split            # noqa: E402
from core.tools.dada_whole_eval import test_sources   # noqa: E402

labels_s8 = json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text())
IDS = {'dota': sorted(load_split(CAP_DEV)), 'tad': sorted(labels_s8), 'dada': test_sources(T2)}
assert len(IDS['dota']) == 569, len(IDS['dota'])
assert len(IDS['tad']) == constants.V2_TADZ_VIDEOS, len(IDS['tad'])
assert len(IDS['dada']) == 370, len(IDS['dada'])
ID_FILE = {s: write_ids(LOCAL / f'{s}_ids.txt', IDS[s]) for s in IDS}
print({s: len(v) for s, v in IDS.items()})
print(f'best.ckpt sha256 {sha256(CKPT)} ({CKPT.stat().st_size / 2**20:.1f} MiB) -- record it with the read-out')

## 5. Integrity (§7.3-1): LaGoVAD reads exactly the arms' CLIP rows

The A2 bakes have no CRN (`s = 1`), so their first 512 columns are the raw CLIP rows. The plain cache LaGoVAD reads must
equal them row for row on every item: DoTA `s1[::3]`, TAD / DADA the s8 caches. Staged to VM disk once (reused below).

In [ ]:
L_PLAIN = {s: stage(PLAIN[s], LOCAL / 'plain' / s, IDS[s]) for s in IDS}
for s in IDS:
    l_bake = stage(BAKE[s], LOCAL / 'bake' / s, IDS[s])
    run([sys.executable, '-m', 'core.tools.zs_lagovad_readout', 'check-inputs', '--plain', L_PLAIN[s],
         '--bake', l_bake, '--ids-file', ID_FILE[s], '--plain-stride', PLAIN_STRIDE[s]])
    shutil.rmtree(l_bake)                 # only the plain rows are scored
print('inputs OK on all three sets')

## 6. DoTA-CAP-dev — protocol B (the table) and protocol A (s8, printed)

In [ ]:
L_DOTA_LAB = stage(DOTA / 'labels_s8', LOCAL / 'dota_labels')
wrote = False
for proto, stride in (('B', constants.V2_E1_STRIDE_BC), ('A', constants.V2_E1_STRIDE_A)):
    out = EVAL / 'dota' / LAG / proto
    if (out / 'protocol_b_readout.json').exists():
        print(f'dota {proto}: already read')
        continue
    local, t0 = LOCAL / 'dota' / proto, time.time()
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--baseline', RUN_NAME, CKPT,
         '--input-dir', L_PLAIN['dota'], '--s1-dir', L_PLAIN['dota'],
         '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--data-dir', L_DOTA_LAB, '--split', CAP_DEV, '--stride', stride, '--out-dir', local,
         '--device', 'auto'], log=LOCAL / f'dota_{proto}.log')
    publish(local, out)
    wrote = True
    print(f'dota {proto}: {(time.time() - t0) / 60:.1f} min')
for proto in ('B', 'A'):
    r = json.loads((EVAL / 'dota' / LAG / proto / 'protocol_b_readout.json').read_text())
    assert r['released_baseline'] and r['two_class'] > 0, r
    print(f"LaGoVAD DoTA-CAP-dev protocol {proto}: macro {r['macro']['mean']:.4f} over {r['two_class']} clips")
if wrote:
    flush_drive()

## 7. TAD — `core.evaluate --baseline-ckpt` (Gate T0's command; micro must be 0.7912 ± 0.001)

In [ ]:
L_TAD = LOCAL / 'TAD'
L_TAD.mkdir(parents=True, exist_ok=True)
for f in TAD.iterdir():                     # json / txt only -- never a frame folder
    if f.is_file() and f.suffix in ('.json', '.txt'):
        shutil.copy2(f, L_TAD / f.name)
out = EVAL / 'tad' / LAG / RUN_NAME
if (out / 'results.json').exists():
    print('tad: already read')
else:
    local = LOCAL / 'tad'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.evaluate', '--baseline-ckpt', CKPT,
         *[x for o in constants.RELEASED_CKPT_OVERRIDES for x in ('--set', o)],
         '--set', f'data.dataset={constants.TAD_DATASET}', '--data-dir', L_TAD, '--clip-dir', L_PLAIN['tad'],
         '--score-norm', 'auto', '--save-scores', '--output-dir', local], log=LOCAL / 'tad.log')
    publish(local, out)
    flush_drive()
r = json.loads((out / 'results.json').read_text())
gap = r['auc'] - constants.ZS_LAGOVAD_T0_MICRO
print(f"LaGoVAD TAD: macro {r['auc_macro']:.4f} | micro {r['auc']:.4f} (Gate T0 {constants.ZS_LAGOVAD_T0_MICRO}, gap {gap:+.5f})")
assert abs(gap) <= constants.ZS_LAGOVAD_T0_TOL, 'released checkpoint does NOT reproduce Gate T0 -- stop, paste this to Claude'

## 8. DADA-2000 — 370 T2-test sources, whole video, every s8 row (`dada_whole_eval --baseline`)

Same definitions as the arms (`Nexar_window/defs.json`), same labels, same rows.

In [ ]:
L_T2_META = LOCAL / 'T2_meta'
L_T2_META.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2_META / constants.META_FILENAME)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in DEFS.glob('*.json'):
    shutil.copy2(f, L_DEFS / f.name)
out = EVAL / 'dada' / LAG
if (out / 'dada_readout.json').exists():
    print('dada: already read')
else:
    local, t0 = LOCAL / 'dada', time.time()
    run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--baseline', RUN_NAME, CKPT,
         '--t2-dir', L_T2_META, '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', L_PLAIN['dada'],
         '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / 'dada.log')
    publish(local, out)
    flush_drive()
    print(f'dada: {(time.time() - t0) / 60:.1f} min')
r = json.loads((out / 'dada_readout.json').read_text())
assert r['released_baseline'] and r['two_class'] > 0, r
print(f"LaGoVAD DADA: macro {r['macro']['mean']:.4f} over {r['two_class']} sources | micro {r['micro']:.4f}")

## 9. Read-out → **paste `REPORTS/zs_lagovad_readout.md` to Claude**

Rule `ZS_TAD_DADA.md` §7, CPU only. It refuses to write if TAD misses Gate T0 or any row covers different items.

In [ ]:
run([sys.executable, '-m', 'core.tools.zs_lagovad_readout', '--lagovad-root', ROOT, '--zs-root', ZS_ROOT,
     '--dota', CONTROL, ARM_DOTA[CONTROL], '--dota', SA2, ARM_DOTA[SA2], '--seed', SEED])
print((REPORTS / 'zs_lagovad_readout.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/zs_lagovad_readout.md to Claude')